## World cup prediction model

## Portfolio Project - Py5.3

### Written by: Shalom Karunwi

#### Last modified: 27-08-2024

### Extracting the fixtures for the 2022 world cup

In [7]:
import pandas as pd
from string import ascii_uppercase as groups
import pickle

In [8]:
# Extracting all the groups from the website
GroupTable = pd.read_html('https://web.archive.org/web/20221115040351/https://en.wikipedia.org/wiki/2022_FIFA_World_Cup')
GroupTable[12]  # The first group is the 12th table by index search
GroupTable[19]  # Each of the tables are seperated by 7 other tables
GroupTable[26]

,Pos,Teamvte,Pld,W,D,L,GF,GA,GD,Pts,Qualification
0,1,Argentina,0,0,0,0,0,0,0,0,Advance to knockout stage
1,2,Saudi Arabia,0,0,0,0,0,0,0,0,Advance to knockout stage
2,3,Mexico,0,0,0,0,0,0,0,0,NaN
3,4,Poland,0,0,0,0,0,0,0,0,NaN


In [9]:
# Renaming each extracted group with appropriate group name
dict_table = {}
for x, i in zip(groups, range(12, 64, 7)):                          # For each table extracted (7 unnecessary tables inbetween each of them)
    df = GroupTable[i]                                              # Iterating through each extracted table  
    df.rename(columns = {df.columns[1] : 'Team'}, inplace = True),  # Renaming the second column to 'Teams'  
    df.drop(columns = df.columns[-1], inplace = True)               # Dropping the 'Qualification' column
    dict_table[f'Group {x}'] = df                                   # Adding each column into a dictionary as Group ("each iteration")

In [10]:
dict_table.keys() # Checking for errors
dict_table['Group E']

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts
0,1,Spain,0,0,0,0,0,0,0,0
1,2,Costa Rica,0,0,0,0,0,0,0,0
2,3,Germany,0,0,0,0,0,0,0,0
3,4,Japan,0,0,0,0,0,0,0,0


In [11]:
with open('groupstage', 'wb') as file:
    pickle.dump(dict_table, file)    # Exporting the dictionary

### Scraping data from all previous world cups

In [13]:
import requests
from bs4 import BeautifulSoup
from pandas.api.types import CategoricalDtype

In [14]:
years1 = [1930, 1934, 1938, 1950, 1954, 1958, 1962,
          1966, 1970, 1974, 1978, 1982, 1986]                                # All previous editions of the world cup

years2 = [1994, 1998, 2002, 2006, 2010, 2014, 2018]                          # 1990 extracts incomplete data so it's handled differently

In [42]:
url = 'https://en.wikipedia.org/wiki/2018_FIFA_World_Cup'                   # Preparing the soup
response = requests.get(url)
file = response.text

In [44]:
soup = BeautifulSoup(file,'lxml')                                           # Saving the soup

In [46]:
matches = soup.find_all('div', class_="footballbox")                        # Extracting the matches played from the soup

In [48]:
# Creating empty lists for each game played
home = []
score = []
away = []

In [50]:
# Appending the scraped information from the website into the empty lists
for match in matches:
    home.append(match.find('th', 'fhome').get_text())
    score.append(match.find('th', 'fscore').get_text())
    away.append(match.find('th', 'faway').get_text())

In [52]:
dict_matches = {'home':home, 'score':score, 'away':away}                    # Saving the lists into a dictionary

In [54]:
# Saving the dictionary into a dataframe
df_games = pd.DataFrame(dict_matches)
df_games['year'] = '2018'
df_games

,home,score,away,year
0,Russia,5–0,Saudi Arabia,2018
1,Egypt,0–1,Uruguay,2018
2,Russia,3–1,Egypt,2018
3,Uruguay,1–0,Saudi Arabia,2018
4,Uruguay,3–0,Russia,2018
...,...,...,...,...
59,Russia,2–2 (a.e.t.),Croatia,2018
60,France,1–0,Belgium,2018
61,Croatia,2–1 (a.e.t.),England,2018
62,Belgium,2–0,England,2018


In [58]:
# Creating a function to use for all previous editions
def all_games_played(year):                                         # The function name
    url = f'https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup'    # The 'f-string' to add the year variable for each year supplied into the function
    response = requests.get(url)
    file = response.text
    
    soup = BeautifulSoup(file,'lxml')   
    
    matches = soup.find_all('div', class_="footballbox")
    
    home = []
    score = []
    away = []
    
    for match in matches:
        home.append(match.find('th', 'fhome').get_text())
        score.append(match.find('th', 'fscore').get_text())
        away.append(match.find('th', 'faway').get_text())
    
    dict_matches = {'home':home, 'score':score, 'away':away} 
    
    df_games = pd.DataFrame(dict_matches)
    df_games['year'] = year                                          # Adding a year column according to the year supplied into the function
    
    return df_games

In [60]:
all_games_played('1990')                                             # Testing out the function

,home,score,away,year
0,Cameroon,2–1 (a.e.t.),Colombia,1990
1,Czechoslovakia,4–1,Costa Rica,1990
2,Brazil,0–1,Argentina,1990
3,West Germany,2–1,Netherlands,1990
4,Republic of Ireland,0–0 (a.e.t.),Romania,1990
5,Italy,2–0,Uruguay,1990
6,Spain,1–2 (a.e.t.),Yugoslavia,1990
7,England,1–0 (a.e.t.),Belgium,1990
8,Argentina,0–0 (a.e.t.),Yugoslavia,1990
9,Republic of Ireland,0–1,Italy,1990


In [62]:
# Creating a dataframe for the missing data in 1990
df_missing_data = pd.DataFrame(
                {'home': ['Italy','United States', 'Italy', 'Austria', 'Italy', 'Austria',
                        'Argentina', 'Soviet Union', 'Argentina', 'Cameroon', 'Argentina', 'Cameroon',
                        'Brazil', 'Costa Rica', 'Brazil', 'Sweden', 'Brazil', 'Sweden',
                        'United Arab Emirates', 'West Germany', 'Yugosalavia', 'West Germany', 'West Germany', 'Yugoslavia',
                        'Belgium', 'Uruguay', 'Belgium', 'South Korea', 'Belgium', 'South Korea',
                        'England', 'Netherlands', 'England', 'Republic of Ireland', 'England', 'Republic of Ireland'], 
                'score': ['1-0', '1-5', '1-0', '0-1', '2-0', '2-1',
                         '0-1', '1-0', '2-0', '2-1', '1-1', '0-4',
                         '2-1', '1-0', '1-0', '1-2', '1-0', '1-2',
                         '0-2', '4-1', '1-0', '5-1', '1-1', '4-1',
                         '2-0', '0-0', '3-1', '1-3', '1-2', '0-1',
                         '1-1', '1-1', '0-0', '0-0', '1-0', '1-1'], 
                'away': ['Austria', 'Czechoslovakia', 'United States', 'Czechoslovakia', 'Czechoslovakia', 'United States',
                        'Cameroon', 'Romania', 'Soviet Union', 'Romania', 'Romania', 'Soviet Union',
                        'Sweden', 'Scotland', 'Costa Rica', 'Scotland', 'Scotland', 'Costa Rica',
                        'Colombia', 'Yugoslavia', 'Colombia', 'United Arab Emirates', 'Colombia', 'United Arab Emirates',
                        'South Korea', 'Spain', 'Uruguay' , 'Spain', 'Spain', 'Uruguay',
                        'Republic of Ireland', 'Egypt', 'Netherlands', 'Egypt', 'Egypt', 'Netherlands']})

In [64]:
df_missing_data['year'] = '1990'       # Adding the year column
df_missing_data                              

,home,score,away,year
0,Italy,1-0,Austria,1990
1,United States,1-5,Czechoslovakia,1990
2,Italy,1-0,United States,1990
3,Austria,0-1,Czechoslovakia,1990
4,Italy,2-0,Czechoslovakia,1990
5,Austria,2-1,United States,1990
6,Argentina,0-1,Cameroon,1990
7,Soviet Union,1-0,Romania,1990
8,Argentina,2-0,Soviet Union,1990
9,Cameroon,2-1,Romania,1990


In [66]:
# Creating a dataframe for the extracted 1990 data
df_1990_2 = all_games_played('1990')

In [67]:
# Appending the extracted data to the missing data
df_1990 = df_missing_data._append(df_1990_2, ignore_index = True)
df_1990

,home,score,away,year
0,Italy,1-0,Austria,1990
1,United States,1-5,Czechoslovakia,1990
2,Italy,1-0,United States,1990
3,Austria,0-1,Czechoslovakia,1990
4,Italy,2-0,Czechoslovakia,1990
5,Austria,2-1,United States,1990
6,Argentina,0-1,Cameroon,1990
7,Soviet Union,1-0,Romania,1990
8,Argentina,2-0,Soviet Union,1990
9,Cameroon,2-1,Romania,1990


In [70]:
# For loops using list comprehension
before_1990 = [all_games_played(year) for year in years1]
after_1990 = [all_games_played(year) for year in years2]           

In [72]:
# Extracting all world cup games data excluding 1990
df_before_1990 = pd.concat(before_1990, ignore_index=True)
df_after_1990 = pd.concat(after_1990, ignore_index = True)

In [74]:
# Adding 1990 data within the extracted data
df_world1 = df_before_1990._append(df_1990, ignore_index=True)
df_world_cup = df_world1._append(df_after_1990, ignore_index=True)
df_world_cup

,home,score,away,year
0,France,4–1,Mexico,1930
1,Argentina,1–0,France,1930
2,Chile,3–0,Mexico,1930
3,Chile,1–0,France,1930
4,Argentina,6–3,Mexico,1930
...,...,...,...,...
896,Russia,2–2 (a.e.t.),Croatia,2018
897,France,1–0,Belgium,2018
898,Croatia,2–1 (a.e.t.),England,2018
899,Belgium,2–0,England,2018


In [76]:
# Extracting the fixtures for the 2022 world cup
url2 = 'https://web.archive.org/web/20221115040351/https://en.wikipedia.org/wiki/2022_FIFA_World_Cup'    
response = requests.get(url2)
file2 = response.text
    
soup2 = BeautifulSoup(file2,'lxml')   
    
matches = soup2.find_all('div', class_="footballbox")
    
home = []
score = []
away = []
    
for match in matches:
        home.append(match.find('th', 'fhome').get_text())
        score.append(match.find('th', 'fscore').get_text())
        away.append(match.find('th', 'faway').get_text())
    
dict_matches = {'home':home, 'score':score, 'away':away} 
    
df_fixtures = pd.DataFrame(dict_matches)
df_fixtures['year'] = 2022
df_fixtures

,home,score,away,year
0,Qatar,Match 1,Ecuador,2022
1,Senegal,Match 2,Netherlands,2022
2,Qatar,Match 18,Senegal,2022
3,Netherlands,Match 19,Ecuador,2022
4,Ecuador,Match 35,Senegal,2022
...,...,...,...,...
59,Winners Match 51,Match 59,Winners Match 52,2022
60,Winners Match 57,Match 61,Winners Match 58,2022
61,Winners Match 59,Match 62,Winners Match 60,2022
62,Losers Match 61,Match 63,Losers Match 62,2022


### Cleaning up the extracted data

In [78]:
df_fifa = pd.read_csv('World_cup_data_2018.csv')
df_fixture = pd.read_csv('World_cup_fixtures_2022.csv')

In [80]:
df_fifa[df_fifa['home'].isnull()]              #Check for null values

,home,score,away,year


In [82]:
df_fifa

,home,score,away,year
0,France,4 1,Mexico,1930
1,Argentina,1 0,France,1930
2,Chile,3 0,Mexico,1930
3,Chile,1 0,France,1930
4,Argentina,6 3,Mexico,1930
...,...,...,...,...
896,Russia,2 2 (a.e.t.),Croatia,2018
897,France,1 0,Belgium,2018
898,Croatia,2 1 (a.e.t.),England,2018
899,Belgium,2 0,England,2018


In [84]:
# Removing all leading and trailing whitespaces
df_fifa['home'] = df_fifa['home'].str.strip()
df_fifa['away'] = df_fifa['away'].str.strip()
df_fixture['home'] = df_fixture['home'].str.strip()
df_fixture['away'] = df_fixture['away'].str.strip()

In [86]:
df_fifa

,home,score,away,year
0,France,4 1,Mexico,1930
1,Argentina,1 0,France,1930
2,Chile,3 0,Mexico,1930
3,Chile,1 0,France,1930
4,Argentina,6 3,Mexico,1930
...,...,...,...,...
896,Russia,2 2 (a.e.t.),Croatia,2018
897,France,1 0,Belgium,2018
898,Croatia,2 1 (a.e.t.),England,2018
899,Belgium,2 0,England,2018


In [88]:
# There is a game in which there was a walkover that needs to be removed

to_delete = df_fifa[df_fifa['score'].str.contains('w/o')].index  # Finding the index of the game
df_fifa.drop(index = to_delete, inplace= True)                        # Deleting the row with the index

In [90]:
# Removing all scores that have (a.e.t)
df_fifa['score'] = df_fifa['score'].str.replace('[(a-z)+.]','',regex=True)

In [92]:
# Removing leading or trailing whitespaces
df_fifa['score'] = df_fifa['score'].str.strip()

In [94]:
# Assigning the home and away goals respectively
df_fifa[['home_goals','away_goals','x']] = df_fifa['score'].str.split(" ", expand=True)

In [96]:
# Dropping the unecessary columns
df_fifa.drop(['score', 'x'], axis=1, inplace=True)

In [98]:
# Checking the datatypes
df_fifa.dtypes

home          object
away          object
year           int64
home_goals    object
away_goals    object
dtype: object

In [100]:
# Assingning appropriate datatypes
df_fifa = df_fifa.astype({'home_goals':int, 'away_goals':int})

In [102]:
# Final check to see if all data was entered properly
years3 = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 
          1978, 1982, 1986, 1994, 1998, 2002, 2006, 2010, 2014, 2018]                             
for year in years3:
    print(year, len(df_fifa[df_fifa['year']==year]))

1930 18
1934 17
1938 18
1950 22
1954 26
1958 35
1962 32
1966 32
1970 32
1974 38
1978 38
1982 52
1986 52
1994 52
1998 64
2002 64
2006 64
2010 64
2014 64
2018 64


### Building the prediction model

In [105]:
from scipy.stats import poisson                                    

In [107]:
group_stage = pickle.load(open('groupstage', 'rb'))                # Group stage tables for 2022 world cup
df_world = pd.read_csv('World_cup_data_cleaned.csv')               # World cup results from all previous editions
df_plays = pd.read_csv('World_cup_fixtures_2022_cleaned.csv')      # Fixtures for all 2022 world cup games

In [109]:
group_stage['Group A']

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts
0,1,Qatar (H),0,0,0,0,0,0,0,0
1,2,Ecuador,0,0,0,0,0,0,0,0
2,3,Senegal,0,0,0,0,0,0,0,0
3,4,Netherlands,0,0,0,0,0,0,0,0


In [111]:
df_home = df_world[['home','home_goals','away_goals']]
df_away = df_world[['away','home_goals','away_goals']]
df_home = df_home.rename(columns = {'home':'Team', 'home_goals':'Goals_scored', 'away_goals':'Goals_conceded'})
df_away = df_away.rename(columns = {'away':'Team', 'home_goals':'Goals_conceded', 'away_goals':'Goals_scored'})

In [113]:
df_away

,Team,Goals_conceded,Goals_scored
0,Mexico,4,1
1,France,1,0
2,Mexico,3,0
3,France,1,0
4,Mexico,6,3
...,...,...,...
895,Croatia,2,2
896,Belgium,1,0
897,England,2,1
898,England,2,0


In [115]:
# Calculating the average goals scored by team over the previous editions
df_avg_goals = pd.concat([df_home,df_away]).groupby('Team').mean()
df_avg_goals

,Goals_scored,Goals_conceded
Team,,
Algeria,1.000000,1.461538
Angola,0.333333,0.666667
Argentina,1.691358,1.148148
Australia,0.812500,1.937500
Austria,1.482759,1.620690
...,...,...
Wales,0.800000,0.800000
West Germany,2.112903,1.241935
Yugosalavia,1.000000,0.000000


In [117]:
# Creating a function to predict each matchup

def prediction(home, away):
    if home in df_avg_goals.index and away in df_avg_goals.index:
        lambda_home = df_avg_goals.at[home,'Goals_scored'] * df_avg_goals.at[away,'Goals_conceded']   # Multiplying goals scored and conceded to serve as the lambda function
        lambda_away = df_avg_goals.at[away,'Goals_scored'] * df_avg_goals.at[home,'Goals_conceded']
        prob_home_win, prob_away_win, prob_draw = 0, 0, 0                                             # Initial states for win, loss and draw
        for x in range(0,11):                                                                         # Range of zero to ten goals for the home team
            for y in range(0,11):                                                                     # Range of zero to ten goals for the away team
                p = poisson.pmf(x, lambda_home) * poisson.pmf(y, lambda_away)                         # Poisson distribution with lamda as above and x&y for homegoals & awaygoals
                if x==y:                                                                              # If draw
                    prob_draw += p
                elif x > y:                                                                           # If home win
                    prob_home_win += p
                else:                                                                                 # If away win
                    prob_away_win += p
        home_points = 3 * prob_home_win + prob_draw                                                   # 3 points for an home win
        away_points = 3 * prob_away_win + prob_draw                                                   # 3 points for an away win
        return (home_points, away_points)                                                             # 1 point each for a draw 
    else:
        return (0, 0)                                                                                 # Return zeros each if team(s) are not in the function (debutants with no prior records)

In [119]:
# Testing out the function
prediction('Argentina', 'Uruguay')

(1.657808914603757, 1.140123223632159)

### The Group Stages

In [122]:
# Structuring the games based on level
df_group_stage = df_plays[:48].copy()        # Group stage games
df_R_16 = df_plays[48:56].copy()             # The round of 16 teams
df_Qf = df_plays[56:60].copy()               # The Quarter final games
df_Sf = df_plays[60:63].copy()               # The semi-final and 3rd place games
df_F = df_plays[63:].copy()                  # The final game

In [124]:
df_group_stage

,home,score,away,year
0,Qatar,Match 1,Ecuador,2022
1,Senegal,Match 2,Netherlands,2022
2,Qatar,Match 18,Senegal,2022
3,Netherlands,Match 19,Ecuador,2022
4,Ecuador,Match 35,Senegal,2022
5,Netherlands,Match 36,Qatar,2022
6,England,Match 3,Iran,2022
7,United States,Match 4,Wales,2022
8,Wales,Match 17,Iran,2022
9,England,Match 20,United States,2022


In [126]:
for group in group_stage:
    team_names = group_stage[group]['Team'].values                                           # Get the name of each team
    df_group_games = df_group_stage[df_group_stage['home'].isin(team_names)]                 # Check if each team is in the group stage D.F and assign to new D.F
    for index, row in df_group_games.iterrows():                                             # For each row in the new D.F
        home, away = row['home'], row['away']                                                # The first team is at home and the second team is away
        home_points, away_points = prediction(home, away)                                    # Use the function to predict a game between home and away
        group_stage[group].loc[group_stage[group]['Team'] == home, 'Pts'] += home_points     # Find the corresponding team in the group table and assign home point
        group_stage[group].loc[group_stage[group]['Team'] == away, 'Pts'] += away_points     # Find the corresponding team in the group table and assign away point

    group_stage[group] = group_stage[group].sort_values('Pts', ascending=False)              # Re-arrange the group table based on points in descending order
    group_stage[group] = group_stage[group][['Team','Pts']]                                  # Show only the teams and their accumulated points
    group_stage[group] = group_stage[group].round(0)                                         # Round each accumulated point to the nearest whole number

C:\Users\s.karunwi\AppData\Local\Temp\ipykernel_17768\88367031.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.8966936]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  group_stage[group].loc[group_stage[group]['Team'] == home, 'Pts'] += home_points     # Find the corresponding team in the group table and assign home point
C:\Users\s.karunwi\AppData\Local\Temp\ipykernel_17768\88367031.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[2.38825101]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  group_stage[group].loc[group_stage[group]['Team'] == home, 'Pts'] += home_points     # Find the corresponding team in the group table and assign home point
C:\Users\s.karunwi\AppData\Local\Temp\ipykernel_17768\88367031.py:7: FutureWarning: Setting an item of incomp

In [128]:
group_stage['Group D']

,Team,Pts
0,France,7.0
2,Denmark,6.0
3,Tunisia,3.0
1,Australia,2.0


In [130]:
df_R_16

,home,score,away,year
48,Winners Group A,Match 49,Runners-up Group B,2022
49,Winners Group C,Match 50,Runners-up Group D,2022
50,Winners Group D,Match 52,Runners-up Group C,2022
51,Winners Group B,Match 51,Runners-up Group A,2022
52,Winners Group E,Match 53,Runners-up Group F,2022
53,Winners Group G,Match 54,Runners-up Group H,2022
54,Winners Group F,Match 55,Runners-up Group E,2022
55,Winners Group H,Match 56,Runners-up Group G,2022


In [132]:
for group in group_stage:
    group_winner = group_stage[group].loc[0, 'Team']                     # Assign the first team as the group winner
    runner_up = group_stage[group].loc[1, 'Team']                        # Assign the second team the runner up

    df_R_16.replace({f'Winners {group}': group_winner,                   # Replace values in the D.F with the appropriate group winner and runner up
                    f'Runners-up {group}': runner_up}, inplace=True)
df_R_16['winner'] = 'TBD' 
df_R_16  

,home,score,away,year,winner
48,Qatar (H),Match 49,Iran,2022,TBD
49,Argentina,Match 50,Australia,2022,TBD
50,France,Match 52,Saudi Arabia,2022,TBD
51,England,Match 51,Ecuador,2022,TBD
52,Spain,Match 53,Canada,2022,TBD
53,Brazil,Match 54,Ghana,2022,TBD
54,Belgium,Match 55,Costa Rica,2022,TBD
55,Portugal,Match 56,Serbia,2022,TBD


In [134]:
def winner(round):
    for index, row in round.iterrows():                      # For each row
        home, away = row['home'], row['away']                # Find the home and away team
        points_home, points_away = prediction(home, away)    # Predict the score between both of them
        if points_home > points_away:                        # If the home team scores more
            winner = home                                    # Home team wins
        else:                                                # If the away team scores more
            winner = away                                    # Away team wins
        round.loc[index, 'winner'] = winner                  # Update the Qf column to reflect the winner
    return round

In [136]:
winner(df_R_16)

,home,score,away,year,winner
48,Qatar (H),Match 49,Iran,2022,Iran
49,Argentina,Match 50,Australia,2022,Argentina
50,France,Match 52,Saudi Arabia,2022,France
51,England,Match 51,Ecuador,2022,England
52,Spain,Match 53,Canada,2022,Spain
53,Brazil,Match 54,Ghana,2022,Brazil
54,Belgium,Match 55,Costa Rica,2022,Belgium
55,Portugal,Match 56,Serbia,2022,Portugal


In [138]:
df_Qf

,home,score,away,year
56,Winners Match 53,Match 58,Winners Match 54,2022
57,Winners Match 49,Match 57,Winners Match 50,2022
58,Winners Match 55,Match 60,Winners Match 56,2022
59,Winners Match 51,Match 59,Winners Match 52,2022


In [140]:
def next_stage(round_1, round_2):
    for index, row in round_1.iterrows():                               # For each row
        winner = round_1.loc[index, 'winner']                           # Find the winner in the winner column
        match = round_1.loc[index, 'score']                             # Find the match number in the score column
        round_2.replace({f'Winners {match}': winner}, inplace=True)     # Replace the home and away columns with the match number winners respectively
    round_2['winner'] = 'TBD'                                           # Add a winner column
    return round_2

### The Quarter-Finals

In [142]:
next_stage(df_R_16, df_Qf)

,home,score,away,year,winner
56,Spain,Match 58,Brazil,2022,TBD
57,Iran,Match 57,Argentina,2022,TBD
58,Belgium,Match 60,Portugal,2022,TBD
59,England,Match 59,France,2022,TBD


In [144]:
winner(df_Qf)

,home,score,away,year,winner
56,Spain,Match 58,Brazil,2022,Brazil
57,Iran,Match 57,Argentina,2022,Argentina
58,Belgium,Match 60,Portugal,2022,Portugal
59,England,Match 59,France,2022,France


### The Semi-Finals

In [146]:
next_stage(df_Qf, df_Sf)

,home,score,away,year,winner
60,Argentina,Match 61,Brazil,2022,TBD
61,France,Match 62,Portugal,2022,TBD
62,Losers Match 61,Match 63,Losers Match 62,2022,TBD


In [148]:
winner(df_Sf)

,home,score,away,year,winner
60,Argentina,Match 61,Brazil,2022,Brazil
61,France,Match 62,Portugal,2022,France
62,Losers Match 61,Match 63,Losers Match 62,2022,Losers Match 62


In [150]:
# 3rd and 4th placed teams
prediction('Argentina', 'Portugal')

(1.4520553471618411, 1.3367139548380762)

### The Final

In [152]:
next_stage(df_Sf, df_F)

,home,score,away,year,winner
63,Brazil,Match 64,France,2022,TBD


In [154]:
winner(df_F)

,home,score,away,year,winner
63,Brazil,Match 64,France,2022,Brazil
